# 🧭 OptiCompromise: Programación de Compromiso Multiobjetivo
### Asignatura: Introducción al Análisis de Decisiones (IAD) | Facultad de Minas – Universidad Nacional de Colombia
**Docente:** Patricia Jaramillo Álvarez, PhD | **Apoyo Académico:** Jaime Andrés Díaz Guerra, MSc  
**Autores:** Fabián Andrés Hurtado Arango (`1036450768`) & Oscar David Arango Garcia (`1011591946`)

---
## Presentación y Modo de Uso
Este cuaderno autónomo implementa la metodología de **Programación de Compromiso** sobre modelos lineales continuos y enteros mixtos (MIP), calculando:
- **Matriz de Pagos** con desempate lexicográfico, **Punto Ideal ($Z^*$)** y **Anti-Ideal / Nadir ($Z_{\text{peor}}$)**.
- **Métrica $L_1$ (Manhattan):** Programación Lineal (HiGHS) con efecto compensatorio.
- **Métrica $L_2$ (Euclidiana):** Programación Cuadrática Convexa (SLSQP + Branch-and-Bound exacto para MIP).
- **Métrica $L_\infty$ (Tchebysheff / Minimax):** Programación Lineal minimax con variable auxiliar $\lambda$ (equidad estricta).
- **Frontera Eficiente de Pareto:** Aproximación 2D/3D y perfil radial de grados de logro $Z'_i = 100\cdot(1 - D_i)$.

> Ejecuta todas las celdas en orden (**Entorno de ejecución → Ejecutar todas**). No requiere clonar repositorios externos.

In [ ]:
# 1. Importación de bibliotecas científicas libres (incluidas por defecto en Google Colab)
import sys
import heapq
import time
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import linprog, minimize
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print(f'✓ Entorno listo | Python {sys.version.split()[0]} | SciPy {scipy.__version__} (HiGHS + SLSQP)')

In [ ]:
# 2. Motor Matemático Autónomo de Programación de Compromiso (LP / QP / MIP + Pareto)
def _build_constraints(model):
    n_vars = len(model["variables"])
    A_ub_list, b_ub_list, A_eq_list, b_eq_list = [], [], [], []
    for c in model.get("constraints", []):
        coeffs = list(c["coefficients"])
        if len(coeffs) < n_vars:
            coeffs += [0.0] * (n_vars - len(coeffs))
        row = np.array(coeffs[:n_vars], dtype=float)
        rhs = float(c["rhs"])
        if c["sense"] == "<=":
            A_ub_list.append(row); b_ub_list.append(rhs)
        elif c["sense"] == ">=":
            A_ub_list.append(-row); b_ub_list.append(-rhs)
        else:
            A_eq_list.append(row); b_eq_list.append(rhs)
    A_ub = np.array(A_ub_list) if A_ub_list else None
    b_ub = np.array(b_ub_list) if b_ub_list else None
    A_eq = np.array(A_eq_list) if A_eq_list else None
    b_eq = np.array(b_eq_list) if b_eq_list else None

    bounds, integrality, has_int = [], [], False
    for v in model["variables"]:
        vt = v.get("var_type", "continuous")
        lb = v.get("lower_bound", 0.0)
        ub = v.get("upper_bound", None)
        if vt == "binary":
            lb = 0.0 if lb is None else max(0.0, float(lb))
            ub = 1.0 if ub is None else min(1.0, float(ub))
            integrality.append(1); has_int = True
        elif vt == "integer":
            integrality.append(1); has_int = True
        else:
            integrality.append(0)
        bounds.append((lb, ub))
    return A_ub, b_ub, A_eq, b_eq, bounds, (integrality if has_int else None)

def _clean_vec(x, bounds, integrality=None):
    out = np.array(x, dtype=float)
    if integrality is not None:
        for j, flag in enumerate(integrality):
            if flag == 1 and abs(out[j] - round(float(out[j]))) < 1e-5:
                out[j] = float(round(float(out[j])))
    for j, (lb, ub) in enumerate(bounds):
        if lb is not None and abs(out[j] - lb) < 1e-9:
            out[j] = lb
        if ub is not None and abs(out[j] - ub) < 1e-9:
            out[j] = ub
    return out

def _is_feasible(x, A_ub, b_ub, A_eq, b_eq, bounds, integrality=None, tol=1e-4):
    if x is None or not np.all(np.isfinite(x)):
        return False
    for j, (lb, ub) in enumerate(bounds):
        if lb is not None and x[j] < lb - tol: return False
        if ub is not None and x[j] > ub + tol: return False
    if A_ub is not None and b_ub is not None and len(b_ub) > 0:
        sc = np.maximum(np.linalg.norm(A_ub, axis=1), np.maximum(np.abs(b_ub), 1.0))
        if np.any((A_ub @ x - b_ub) / sc > tol): return False
    if A_eq is not None and b_eq is not None and len(b_eq) > 0:
        sc = np.maximum(np.linalg.norm(A_eq, axis=1), np.maximum(np.abs(b_eq), 1.0))
        if np.any(np.abs(A_eq @ x - b_eq) / sc > tol): return False
    if integrality is not None:
        for j, flag in enumerate(integrality):
            if flag == 1 and abs(x[j] - round(float(x[j]))) > 1e-4: return False
    return True

def _solve_qp(alpha, beta, w, A_ub, b_ub, A_eq, b_eq, bounds, starts_in):
    n_vars = len(bounds)
    for lb, ub in bounds:
        if lb is not None and ub is not None and lb > ub + 1e-9:
            return None, float("inf")
    def obj(x):
        d = alpha @ x + beta
        return float(np.sum(w * (d ** 2)))
    def jac(x):
        d = alpha @ x + beta
        return 2.0 * (alpha.T @ (w * d))
    cons = []
    if A_ub is not None and b_ub is not None:
        for i in range(len(b_ub)):
            row, rhs = A_ub[i], float(b_ub[i])
            sc = float(max(np.linalg.norm(row), abs(rhs), 1.0))
            rs, bs = row / sc, rhs / sc
            cons.append({"type": "ineq", "fun": lambda x, r=rs, b=bs: b - float(np.dot(r, x)), "jac": lambda x, r=rs: -r})
    if A_eq is not None and b_eq is not None:
        for i in range(len(b_eq)):
            row, rhs = A_eq[i], float(b_eq[i])
            sc = float(max(np.linalg.norm(row), abs(rhs), 1.0))
            rs, bs = row / sc, rhs / sc
            cons.append({"type": "eq", "fun": lambda x, r=rs, b=bs: float(np.dot(r, x)) - b, "jac": lambda x, r=rs: r})

    starts = []
    for c in starts_in:
        if c is not None and len(c) == n_vars and np.all(np.isfinite(c)):
            p = np.array(c, dtype=float)
            for j, (lb, ub) in enumerate(bounds):
                if lb is not None: p[j] = max(p[j], lb)
                if ub is not None: p[j] = min(p[j], ub)
            starts.append(p)
    if not starts:
        starts.append(np.zeros(n_vars, dtype=float))

    best_x, best_val = None, float("inf")
    for s in starts:
        if _is_feasible(s, A_ub, b_ub, A_eq, b_eq, bounds):
            v = obj(s)
            if v < best_val: best_val, best_x = v, s
    for s in starts[:2]:
        res = minimize(obj, s, jac=jac, method="SLSQP", bounds=bounds, constraints=cons, options={"maxiter": 400, "ftol": 1e-11})
        if res.x is not None and _is_feasible(res.x, A_ub, b_ub, A_eq, b_eq, bounds):
            v = obj(res.x)
            if v < best_val: best_val, best_x = v, np.array(res.x, dtype=float)
    if best_x is None:
        feas = linprog(np.zeros(n_vars), A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
        if feas.success and feas.x is not None:
            best_x = np.array(feas.x, dtype=float); best_val = obj(best_x)
            res = minimize(obj, best_x, jac=jac, method="SLSQP", bounds=bounds, constraints=cons, options={"maxiter": 400, "ftol": 1e-10})
            if res.x is not None and _is_feasible(res.x, A_ub, b_ub, A_eq, b_eq, bounds):
                v = obj(res.x)
                if v < best_val: best_val, best_x = v, np.array(res.x, dtype=float)
    return best_x, best_val

def resolver_opticompromise(model, weights=None, compute_pareto=True):
    n_vars = len(model["variables"])
    n_objs = len(model["objectives"])
    A_ub, b_ub, A_eq, b_eq, bounds, integ = _build_constraints(model)

    obj_dirs = []
    obj_coeffs = []
    for o in model["objectives"]:
        c = list(o["coefficients"])
        if len(c) < n_vars: c += [0.0] * (n_vars - len(c))
        arr = np.array(c[:n_vars], dtype=float)
        obj_coeffs.append(arr)
        obj_dirs.append(-arr if o["sense"] == "maximize" else arr)

    payoff = np.zeros((n_objs, n_objs), dtype=float)
    ind_sols = []
    for k in range(n_objs):
        res = linprog(obj_dirs[k], A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, integrality=integ, method="highs")
        if not res.success:
            raise ValueError(f"Error resolviendo objetivo {model['objectives'][k]['name']}: {res.message}")
        x_k = np.array(res.x, dtype=float)
        if n_objs >= 2:
            z_k = float(np.dot(obj_dirs[k], x_k))
            sec = sum((obj_dirs[j] / np.linalg.norm(obj_dirs[j])) for j in range(n_objs) if j != k and np.linalg.norm(obj_dirs[j]) > 1e-12)
            if isinstance(sec, np.ndarray) and np.linalg.norm(sec) > 1e-12:
                A_eq_lex = np.vstack([A_eq, obj_dirs[k][None, :]]) if A_eq is not None else obj_dirs[k][None, :]
                b_eq_lex = np.concatenate([b_eq, [z_k]]) if b_eq is not None else np.array([z_k])
                r_lex = linprog(sec, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq_lex, b_eq=b_eq_lex, bounds=bounds, integrality=integ, method="highs")
                if r_lex.success and r_lex.x is not None and float(np.dot(obj_dirs[k], r_lex.x)) <= z_k + 1e-9 * max(1.0, abs(z_k)):
                    x_k = np.array(r_lex.x, dtype=float)
        x_k = _clean_vec(x_k, bounds, integ)
        ind_sols.append(x_k)
        for i in range(n_objs):
            payoff[k, i] = float(np.dot(obj_coeffs[i], x_k))

    ideal = np.zeros(n_objs)
    anti_ideal = np.zeros(n_objs)
    for i, o in enumerate(model["objectives"]):
        col = payoff[:, i]
        ideal[i] = float(np.max(col)) if o["sense"] == "maximize" else float(np.min(col))
        anti_ideal[i] = float(np.min(col)) if o["sense"] == "maximize" else float(np.max(col))
        if np.isclose(ideal[i], anti_ideal[i]):
            dlt = abs(ideal[i]) * 0.05 if not np.isclose(ideal[i], 0.0) else 1.0
            anti_ideal[i] = ideal[i] - dlt if o["sense"] == "maximize" else ideal[i] + dlt

    alpha = np.zeros((n_objs, n_vars))
    beta = np.zeros(n_objs)
    for i, o in enumerate(model["objectives"]):
        rng = (ideal[i] - anti_ideal[i]) if o["sense"] == "maximize" else (anti_ideal[i] - ideal[i])
        if np.isclose(rng, 0.0): rng = 1.0
        if o["sense"] == "maximize":
            alpha[i] = -obj_coeffs[i] / rng; beta[i] = ideal[i] / rng
        else:
            alpha[i] = obj_coeffs[i] / rng; beta[i] = -ideal[i] / rng

    if weights is None or len(weights) != n_objs:
        w = np.ones(n_objs) / n_objs
    else:
        w = np.maximum(0.0, np.array(weights, dtype=float))
        w = w / w.sum() if w.sum() > 0 else np.ones(n_objs) / n_objs

    def solve_for_w(w_vec):
        w_eff = np.where(w_vec <= 1e-9, 1e-6, w_vec) if np.any(w_vec <= 1e-9) else w_vec
        # L1
        r1 = linprog(np.sum(w_eff[:, None] * alpha, axis=0), A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, integrality=integ, method="highs")
        x1 = _clean_vec(r1.x, bounds, integ)
        # L_inf
        c_inf = np.zeros(n_vars + 1)
        if np.any(w_vec <= 1e-9): c_inf[:n_vars] = 1e-6 * np.sum(alpha, axis=0)
        c_inf[-1] = 1.0
        A_m = np.hstack([w_vec[:, None] * alpha, -np.ones((n_objs, 1))])
        b_m = -w_vec * beta
        A_u_tot = np.vstack([np.hstack([A_ub, np.zeros((len(A_ub), 1))]), A_m]) if A_ub is not None else A_m
        b_u_tot = np.concatenate([b_ub, b_m]) if b_ub is not None else b_m
        A_e_tot = np.hstack([A_eq, np.zeros((len(A_eq), 1))]) if A_eq is not None else None
        ri = linprog(c_inf, A_ub=A_u_tot, b_ub=b_u_tot, A_eq=A_e_tot, b_eq=b_eq, bounds=list(bounds) + [(0.0, None)], integrality=(list(integ) + [0] if integ is not None else None), method="highs")
        xi = _clean_vec(ri.x[:n_vars], bounds, integ)
        # L2
        cands = [0.5 * (x1 + xi), x1, xi, np.mean(ind_sols, axis=0)]
        if integ is None:
            x2, _ = _solve_qp(alpha, beta, w_eff, A_ub, b_ub, A_eq, b_eq, bounds, cands)
            x2 = _clean_vec(x2, bounds, None)
        else:
            int_idx = [j for j, fl in enumerate(integ) if fl == 1]
            best_x, best_v = None, float("inf")
            for c in cands + ind_sols:
                sn = np.array(c, dtype=float)
                for j in int_idx: sn[j] = round(float(sn[j]))
                if _is_feasible(sn, A_ub, b_ub, A_eq, b_eq, bounds, integ):
                    v = float(np.sum(w_eff * ((alpha @ sn + beta) ** 2)))
                    if v < best_v: best_v, best_x = v, sn
            rx, rlb = _solve_qp(alpha, beta, w_eff, A_ub, b_ub, A_eq, b_eq, bounds, cands)
            if rx is not None:
                hp = [(rlb, 0, bounds, rx)]
                cnt, exp, t0 = 0, 0, time.monotonic()
                while hp and exp < 400 and (time.monotonic() - t0) < 0.5:
                    lb_v, _, nb, nx = heapq.heappop(hp)
                    if lb_v >= best_v - 1e-7: continue
                    exp += 1
                    bj, mf = -1, 0.0
                    for j in int_idx:
                        fr = abs(nx[j] - round(float(nx[j])))
                        if fr > 1e-4 and fr > mf: mf, bj = fr, j
                    if bj == -1:
                        sn = np.array(nx, dtype=float)
                        for j in int_idx: sn[j] = round(float(sn[j]))
                        if _is_feasible(sn, A_ub, b_ub, A_eq, b_eq, bounds, integ):
                            v = float(np.sum(w_eff * ((alpha @ sn + beta) ** 2)))
                            if v < best_v: best_v, best_x = v, sn
                        continue
                    fl, ce = float(np.floor(nx[bj])), float(np.ceil(nx[bj]))
                    ol, ou = nb[bj]
                    for cl, cu in [(ol, fl if ou is None else min(ou, fl)), (ce if ol is None else max(ol, ce), ou)]:
                        if cl is not None and cu is not None and cl > cu + 1e-9: continue
                        cb = list(nb); cb[bj] = (cl, cu)
                        cx, clb = _solve_qp(alpha, beta, w_eff, A_ub, b_ub, A_eq, b_eq, cb, [nx])
                        if cx is not None and clb < best_v - 1e-7:
                            cnt += 1; heapq.heappush(hp, (clb, cnt, cb, cx))
            x2 = _clean_vec(best_x, bounds, integ)
        return x1, x2, xi

    x1, x2, xi = solve_for_w(w)
    pareto_pts = []
    if compute_pareto:
        w_grid = []
        if n_objs == 2:
            w_grid = [np.array([t, 1.0 - t]) for t in np.linspace(0.0, 1.0, 17)]
        elif n_objs == 3:
            for i in range(7):
                for j in range(7 - i):
                    w1, w2 = i / 6.0, j / 6.0
                    w_grid.append(np.array([w1, w2, max(0.0, 1.0 - w1 - w2)]))
        else:
            rng_p = np.random.default_rng(42)
            w_grid = [np.ones(n_objs) / n_objs] + [rng_p.dirichlet(np.ones(n_objs)) for _ in range(20)]
        raw_z = []
        for wg in w_grid:
            p1, p2, pi = solve_for_w(wg)
            for px in (p1, p2, pi):
                raw_z.append([float(np.dot(obj_coeffs[k], px)) for k in range(n_objs)])
        senses = [o["sense"] for o in model["objectives"]]
        def dom(a, b):
            be, sb = True, False
            for va, vb, s in zip(a, b, senses):
                if s == "maximize":
                    if vb < va - 1e-6: be = False; break
                    if vb > va + 1e-6: sb = True
                else:
                    if vb > va + 1e-6: be = False; break
                    if vb < va - 1e-6: sb = True
            return be and sb
        nd = [p for p in raw_z if not any(dom(p, q) for q in raw_z)]
        for p in nd:
            if not any(np.allclose(p, s, atol=1e-3, rtol=5e-3) for s in pareto_pts):
                pareto_pts.append(p)
        if n_objs == 2:
            pareto_pts.sort(key=lambda p: p[0])

    return {
        "payoff": payoff, "ideal": ideal, "anti_ideal": anti_ideal,
        "alpha": alpha, "beta": beta, "weights": w,
        "L1_x": x1, "L2_x": x2, "Linf_x": xi,
        "pareto": pareto_pts, "obj_coeffs": obj_coeffs
    }

print("✓ Motor OptiCompromise cargado exitosamente.")


In [ ]:
# 3. Casos de Validación Oficiales del Curso y Modelos Bonus
MODELOS_EJEMPLO = {
  "1. Celdas Fotovoltaicas (Clase 15 · Curso)": {
    "name": "Planificación de Celdas Fotovoltaicas (Clase 15)",
    "description": "Problema biobjetivo del curso: maximizar utilidad económica vs minimizar generación de contaminantes.",
    "variables": [
      {
        "name": "x1 (Celdas Fotov.)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 300.0
      },
      {
        "name": "x2 (Celdas GP)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 200.0
      }
    ],
    "objectives": [
      {
        "name": "Utilidad Total (USD)",
        "sense": "maximize",
        "coefficients": [
          1000.0,
          3000.0
        ]
      },
      {
        "name": "Emisión Contaminantes (kg)",
        "sense": "minimize",
        "coefficients": [
          1.0,
          2.0
        ]
      }
    ],
    "constraints": [
      {
        "name": "Capacidad Máxima de Ensamble",
        "coefficients": [
          1.0,
          1.0
        ],
        "sense": "<=",
        "rhs": 400.0
      },
      {
        "name": "Demanda Mínima de Ingreso",
        "coefficients": [
          1000.0,
          3000.0
        ],
        "sense": ">=",
        "rhs": 300000.0
      }
    ],
    "weights": [
      0.5,
      0.5
    ]
  },
  "2. Problema de Transporte (Taller 3 · 12 Variables)": {
    "name": "Problema de Transporte Multiobjetivo (Taller 3)",
    "description": "Distribución desde 3 orígenes (O1, O2, O3) hacia 4 destinos (D1, D2, D3, D4), minimizando costo total de flete y minimizando riesgo en rutas.",
    "variables": [
      {
        "name": "x_1_1 (O1 → D1)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_1_2 (O1 → D2)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_1_3 (O1 → D3)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_1_4 (O1 → D4)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_2_1 (O2 → D1)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_2_2 (O2 → D2)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_2_3 (O2 → D3)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_2_4 (O2 → D4)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_3_1 (O3 → D1)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_3_2 (O3 → D2)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_3_3 (O3 → D3)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x_3_4 (O3 → D4)",
        "var_type": "continuous",
        "lower_bound": 0.0
      }
    ],
    "objectives": [
      {
        "name": "Costo de Transporte (COP)",
        "sense": "minimize",
        "coefficients": [
          4.0,
          3.0,
          3.0,
          4.0,
          5.0,
          2.0,
          1.0,
          5.0,
          7.0,
          10.0,
          8.0,
          9.0
        ]
      },
      {
        "name": "Riesgo Global de Transporte",
        "sense": "minimize",
        "coefficients": [
          6.0,
          0.0,
          1.0,
          5.0,
          2.0,
          4.0,
          0.0,
          2.0,
          3.0,
          1.0,
          2.0,
          1.0
        ]
      }
    ],
    "constraints": [
      {
        "name": "Oferta Origen O1",
        "coefficients": [
          1.0,
          1.0,
          1.0,
          1.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0
        ],
        "sense": "<=",
        "rhs": 800.0
      },
      {
        "name": "Oferta Origen O2",
        "coefficients": [
          0.0,
          0.0,
          0.0,
          0.0,
          1.0,
          1.0,
          1.0,
          1.0,
          0.0,
          0.0,
          0.0,
          0.0
        ],
        "sense": "<=",
        "rhs": 1200.0
      },
      {
        "name": "Oferta Origen O3",
        "coefficients": [
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          0.0,
          1.0,
          1.0,
          1.0,
          1.0
        ],
        "sense": "<=",
        "rhs": 1500.0
      },
      {
        "name": "Demanda Destino D1",
        "coefficients": [
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0
        ],
        "sense": ">=",
        "rhs": 500.0
      },
      {
        "name": "Demanda Destino D2",
        "coefficients": [
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0
        ],
        "sense": ">=",
        "rhs": 700.0
      },
      {
        "name": "Demanda Destino D3",
        "coefficients": [
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0
        ],
        "sense": ">=",
        "rhs": 900.0
      },
      {
        "name": "Demanda Destino D4",
        "coefficients": [
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0,
          0.0,
          0.0,
          0.0,
          1.0
        ],
        "sense": ">=",
        "rhs": 1400.0
      }
    ],
    "weights": [
      0.5,
      0.5
    ]
  },
  "3. Sostenibilidad Triobjetivo 3D (Bonus 3Z)": {
    "name": "Optimización Sostenible Triobjetivo 3D (Bonus)",
    "description": "Problema con 3 dimensiones de impacto: maximizar retorno económico, minimizar impacto ambiental y maximizar beneficio social.",
    "variables": [
      {
        "name": "x1 (Línea Rentable)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x2 (Línea Social)",
        "var_type": "continuous",
        "lower_bound": 0.0
      },
      {
        "name": "x3 (Línea Intensiva)",
        "var_type": "continuous",
        "lower_bound": 0.0
      }
    ],
    "objectives": [
      {
        "name": "Retorno Económico (kUSD)",
        "sense": "maximize",
        "coefficients": [
          3.0,
          1.0,
          2.0
        ]
      },
      {
        "name": "Impacto Ambiental (CO2 eq)",
        "sense": "minimize",
        "coefficients": [
          2.0,
          1.0,
          4.0
        ]
      },
      {
        "name": "Beneficio Social (Puntos)",
        "sense": "maximize",
        "coefficients": [
          1.0,
          3.0,
          1.0
        ]
      }
    ],
    "constraints": [
      {
        "name": "Presupuesto de Capital",
        "coefficients": [
          1.0,
          1.0,
          1.0
        ],
        "sense": "<=",
        "rhs": 100.0
      },
      {
        "name": "Capacidad Técnica Operativa A",
        "coefficients": [
          2.0,
          1.0,
          0.0
        ],
        "sense": "<=",
        "rhs": 120.0
      },
      {
        "name": "Capacidad Técnica Operativa B",
        "coefficients": [
          0.0,
          1.0,
          2.0
        ],
        "sense": "<=",
        "rhs": 140.0
      }
    ],
    "weights": [
      0.3333,
      0.3333,
      0.3334
    ]
  },
  "4. Producción Mixta MIP (Bonus Entera/Binaria)": {
    "name": "Planificación Entera y Binaria (MIP Bonus)",
    "description": "Optimización de rendimiento vs costo operativo con variables continuas (lotes), enteras (máquinas) y binaria (apertura de planta).",
    "variables": [
      {
        "name": "x1",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x2",
        "var_type": "integer",
        "lower_bound": 0.0,
        "upper_bound": 20.0
      },
      {
        "name": "y1",
        "var_type": "binary",
        "lower_bound": 0.0,
        "upper_bound": 1.0
      }
    ],
    "objectives": [
      {
        "name": "Rendimiento (USD)",
        "sense": "maximize",
        "coefficients": [
          40.0,
          70.0,
          150.0
        ]
      },
      {
        "name": "Costo Operativo (USD)",
        "sense": "minimize",
        "coefficients": [
          10.0,
          25.0,
          80.0
        ]
      }
    ],
    "constraints": [
      {
        "name": "Capacidad Planta",
        "coefficients": [
          1.0,
          2.0,
          -30.0
        ],
        "sense": "<=",
        "rhs": 0.0
      },
      {
        "name": "Presupuesto Global",
        "coefficients": [
          10.0,
          25.0,
          80.0
        ],
        "sense": "<=",
        "rhs": 500.0
      },
      {
        "name": "Demanda Mínima",
        "coefficients": [
          1.0,
          1.0,
          0.0
        ],
        "sense": ">=",
        "rhs": 15.0
      }
    ],
    "weights": [
      0.6,
      0.4
    ]
  },
  "5. Flota de Vehículos Eléctricos (6 Autos, 4 Objetivos)": {
    "name": "Adquisición de Flota de Vehículos Eléctricos (Caso Taller IAD)",
    "description": "Evaluación multicriterio para adquisición de flota entre 6 alternativas de vehículos 100% eléctricos: BYD Seal, Tesla Model 3, Hyundai Ioniq 5, Kia EV6, Volvo EX30 y Chevrolet Equinox EV.",
    "variables": [
      {
        "name": "x1 (BYD Seal)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x2 (Tesla Model 3)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x3 (Hyundai Ioniq 5)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x4 (Kia EV6)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x5 (Volvo EX30)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      },
      {
        "name": "x6 (Chevrolet Equinox EV)",
        "var_type": "continuous",
        "lower_bound": 0.0,
        "upper_bound": 50.0
      }
    ],
    "objectives": [
      {
        "name": "Costo Total Flota (USD)",
        "sense": "minimize",
        "coefficients": [
          44000.0,
          46500.0,
          49000.0,
          48000.0,
          42000.0,
          43500.0
        ]
      },
      {
        "name": "Autonomía Total (km)",
        "sense": "maximize",
        "coefficients": [
          570.0,
          630.0,
          520.0,
          540.0,
          480.0,
          510.0
        ]
      },
      {
        "name": "Tiempo Promedio Carga (min)",
        "sense": "minimize",
        "coefficients": [
          32.0,
          27.0,
          18.0,
          18.0,
          26.0,
          30.0
        ]
      },
      {
        "name": "Índice Seguridad Flota (pts)",
        "sense": "maximize",
        "coefficients": [
          9.0,
          9.5,
          9.6,
          9.7,
          10.0,
          9.3
        ]
      }
    ],
    "constraints": [
      {
        "name": "Tamaño Requerido de Flota",
        "coefficients": [
          1.0,
          1.0,
          1.0,
          1.0,
          1.0,
          1.0
        ],
        "sense": "==",
        "rhs": 50.0
      },
      {
        "name": "Capacidad Mínima Maletero (L)",
        "coefficients": [
          400.0,
          594.0,
          527.0,
          520.0,
          318.0,
          748.0
        ],
        "sense": ">=",
        "rhs": 24000.0
      },
      {
        "name": "Presupuesto Máximo Inversión (USD)",
        "coefficients": [
          44000.0,
          46500.0,
          49000.0,
          48000.0,
          42000.0,
          43500.0
        ],
        "sense": "<=",
        "rhs": 2350000.0
      }
    ],
    "weights": [
      0.3,
      0.25,
      0.25,
      0.2
    ]
  }
}
print(f'✓ {len(MODELOS_EJEMPLO)} modelos precargados listos.')


In [ ]:
# 4. Panel Interactivo en Colab (Deslizadores dinámicos para todos los objetivos + Gráficas)
dropdown_modelo = widgets.Dropdown(
    options=list(MODELOS_EJEMPLO.keys()),
    description="Modelo:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="560px")
)
weights_box = widgets.VBox()
output_dashboard = widgets.Output()
sliders_refs = []

def render_dashboard(*args):
    m_data = MODELOS_EJEMPLO[dropdown_modelo.value]
    raw_w = [s.value for s in sliders_refs]
    with output_dashboard:
        clear_output(wait=True)
        res = resolver_opticompromise(m_data, weights=raw_w, compute_pareto=True)
        w_norm = res["weights"]
        obj_names = [o["name"] for o in m_data["objectives"]]
        var_names = [v["name"] for v in m_data["variables"]]

        display(HTML(f"<h3>📊 1. Matriz de Pagos y Puntos de Referencia ({m_data['name']})</h3>"))
        df_payoff = pd.DataFrame(res["payoff"], index=[f"Óptimo {n}" for n in obj_names], columns=obj_names)
        df_payoff.loc["Punto Ideal (Z*)"] = res["ideal"]
        df_payoff.loc["Anti-Ideal / Nadir"] = res["anti_ideal"]
        display(df_payoff.round(2))

        def eval_x(x):
            z = [float(np.dot(res["obj_coeffs"][i], x)) for i in range(len(obj_names))]
            d = np.maximum(0.0, res["alpha"] @ x + res["beta"])
            ach = 100.0 * (1.0 - d)
            return z, ach, d

        rows = []
        for label, x_sol in [("L1 (Manhattan)", res["L1_x"]), ("L2 (Euclidiana)", res["L2_x"]), ("L_inf (Minimax)", res["Linf_x"])]:
            z, ach, d = eval_x(x_sol)
            if "L1" in label: dp = float(np.sum(w_norm * d))
            elif "L2" in label: dp = float(np.sqrt(np.sum(w_norm * (d ** 2))))
            else: dp = float(np.max(w_norm * d))
            active_vars = ", ".join([f"{var_names[j]}={x_sol[j]:.2f}" for j in range(len(x_sol)) if abs(x_sol[j]) > 1e-3])
            item = {"Métrica": label, "Distancia D_p": round(dp, 4), "Variables x*": active_vars or "Todas en 0"}
            for idx, oname in enumerate(obj_names):
                item[f"{oname}"] = round(z[idx], 2)
                item[f"% Logro Z'{idx+1}"] = f"{ach[idx]:.2f}%"
            rows.append(item)
        display(HTML(f"<h3>🎯 2. Soluciones Compromiso (Pesos normalizados: {[round(float(v), 3) for v in w_norm]})</h3>"))
        display(pd.DataFrame(rows))

        n_objs = len(obj_names)
        is_mip = any(v.get("var_type", "continuous") != "continuous" for v in m_data["variables"])
        z1, _a1, _ = eval_x(res["L1_x"])
        z2, _a2, _ = eval_x(res["L2_x"])
        zi, _ai, _ = eval_x(res["Linf_x"])

        if n_objs == 2:
            plt.figure(figsize=(8.5, 4.8))
            px = [p[0] for p in res["pareto"]]; py = [p[1] for p in res["pareto"]]
            if px:
                if is_mip:
                    plt.scatter(px, py, color="#0F766E", s=40, alpha=0.75, label="Frontera Discreta (MIP)")
                else:
                    plt.plot(px, py, color="#E11D48", lw=2.2, alpha=0.85, label="Frontera de Pareto")
            plt.scatter(res["ideal"][0], res["ideal"][1], color="#D97706", s=220, marker="*", label="Ideal (Z*)", zorder=6)
            plt.scatter(res["anti_ideal"][0], res["anti_ideal"][1], color="#BE123C", s=110, marker="X", label="Nadir", zorder=6)
            plt.scatter(z1[0], z1[1], color="#2563EB", s=120, marker="s", label="L1", zorder=7)
            plt.scatter(z2[0], z2[1], color="#334155", s=120, marker="o", label="L2", zorder=7)
            plt.scatter(zi[0], zi[1], color="#E11D48", s=130, marker="D", label="L_inf", zorder=7)
            plt.title(f"Espacio de Objetivos: {m_data['name']}", fontweight="bold")
            plt.xlabel(obj_names[0]); plt.ylabel(obj_names[1])
            plt.grid(True, linestyle="--", alpha=0.5); plt.legend(loc="best"); plt.tight_layout(); plt.show()
        elif n_objs == 3:
            fig = plt.figure(figsize=(8.5, 5.5))
            ax = fig.add_subplot(111, projection="3d")
            if res["pareto"]:
                ax.scatter([p[0] for p in res["pareto"]], [p[1] for p in res["pareto"]], [p[2] for p in res["pareto"]], color="#0F766E", s=24, alpha=0.5, label="Frontera Pareto 3D")
            ax.scatter([res["ideal"][0]], [res["ideal"][1]], [res["ideal"][2]], color="#D97706", s=180, marker="*", label="Ideal (Z*)")
            ax.scatter([res["anti_ideal"][0]], [res["anti_ideal"][1]], [res["anti_ideal"][2]], color="#BE123C", s=100, marker="X", label="Nadir")
            ax.scatter([z1[0]], [z1[1]], [z1[2]], color="#2563EB", s=110, marker="s", label="L1")
            ax.scatter([z2[0]], [z2[1]], [z2[2]], color="#334155", s=110, marker="o", label="L2")
            ax.scatter([zi[0]], [zi[1]], [zi[2]], color="#E11D48", s=120, marker="D", label="L_inf")
            ax.set_xlabel(obj_names[0]); ax.set_ylabel(obj_names[1]); ax.set_zlabel(obj_names[2])
            ax.set_title(f"Espacio 3D: {m_data['name']}", fontweight="bold")
            ax.legend(loc="best"); plt.tight_layout(); plt.show()
        else:
            N = n_objs
            angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0.0]
            fig, ax = plt.subplots(figsize=(6.5, 6.5), subplot_kw=dict(polar=True))
            plt.xticks(angles[:-1], obj_names, size=9, fontweight="bold")
            plt.ylim(0, 108)
            ax.plot(angles, [100]*(N+1), lw=1.5, ls="--", color="#D97706", label="Ideal (100%)")
            for lbl, col, x_s in [("L1", "#2563EB", res["L1_x"]), ("L2", "#334155", res["L2_x"]), ("L_inf", "#E11D48", res["Linf_x"])]:
                _, a_s, _ = eval_x(x_s)
                vals = list(np.clip(a_s, 0, 100)) + [float(np.clip(a_s[0], 0, 100))]
                ax.plot(angles, vals, lw=2, color=col, label=lbl); ax.fill(angles, vals, col, alpha=0.1)
            plt.title(f"Perfil Multicriterio (% Logro): {m_data['name']}", fontweight="bold", y=1.08)
            plt.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1)); plt.tight_layout(); plt.show()

def rebuild_sliders(*args):
    global sliders_refs
    m_data = MODELOS_EJEMPLO[dropdown_modelo.value]
    n_objs = len(m_data["objectives"])
    def_w = m_data.get("weights") or [1.0 / n_objs] * n_objs
    sliders_refs = []
    for i, o in enumerate(m_data["objectives"]):
        sl = widgets.FloatSlider(
            value=float(def_w[i]), min=0.0, max=1.0, step=0.05,
            description=f"w{i+1} ({o['name'][:22]}):",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="520px")
        )
        sl.observe(render_dashboard, names="value")
        sliders_refs.append(sl)
    weights_box.children = tuple(sliders_refs)
    render_dashboard()

dropdown_modelo.observe(rebuild_sliders, names="value")
display(HTML("<h2>🚀 OptiCompromise — Cuaderno Interactivo UNAL</h2>"))
display(dropdown_modelo, weights_box, output_dashboard)
rebuild_sliders()
